# 第18章 回归

这个 notebook 用一个小型光度红移教学例子，把回归流程拆成可读的步骤：切分数据、手写线性回归 baseline、比较二次特征、检查训练/测试误差和残差。

## 学习目标

- 区分连续量预测与分类任务
- 手写一个最小线性回归 baseline
- 计算 MAE、RMSE 和残差
- 比较训练误差与测试误差
- 理解多项式特征为什么既有帮助也有风险

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/photometric_redshift_demo.csv')
with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    for key in ('u_g', 'g_r', 'r_i', 'i_z', 'specz'):
        row[key] = float(row[key])

specz_values = [row['specz'] for row in rows]
print('loaded rows:', len(rows))
print('redshift range:', (min(specz_values), max(specz_values)))
print('Python version:', platform.python_version())


## 一个固定的教学切分

为了让 notebook 的输出稳定，我们使用固定索引划分测试集。真实项目通常需要更系统的交叉验证和更大的样本。

In [ ]:
test_indices = {2, 6, 10}
train_rows = [row for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row for index, row in enumerate(rows) if index in test_indices]

print('train:', len(train_rows), 'test:', len(test_rows))
print('test galaxy ids:', [row['galaxy_id'] for row in test_rows])


## 最小线性回归 baseline

这里先只使用一个颜色特征 `g-r`。这不是最佳科学方案，而是最容易理解的一维教学起点。

In [ ]:
def fit_line(xs, ys):
    x_mean = sum(xs) / len(xs)
    y_mean = sum(ys) / len(ys)
    numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    denominator = sum((x - x_mean) ** 2 for x in xs)
    slope = numerator / denominator
    intercept = y_mean - slope * x_mean
    return slope, intercept

def predict_line(x_value, slope, intercept):
    return slope * x_value + intercept

def mean_absolute_error(y_true, y_pred):
    return sum(abs(a - b) for a, b in zip(y_true, y_pred)) / len(y_true)

def root_mean_squared_error(y_true, y_pred):
    mse = sum((a - b) ** 2 for a, b in zip(y_true, y_pred)) / len(y_true)
    return math.sqrt(mse)

x_train = [row['g_r'] for row in train_rows]
y_train = [row['specz'] for row in train_rows]
x_test = [row['g_r'] for row in test_rows]
y_test = [row['specz'] for row in test_rows]

slope, intercept = fit_line(x_train, y_train)
train_predictions = [predict_line(x, slope, intercept) for x in x_train]
test_predictions = [predict_line(x, slope, intercept) for x in x_test]

print('slope =', round(slope, 3))
print('intercept =', round(intercept, 3))
print('test predictions =', [round(value, 3) for value in test_predictions])


In [ ]:
line_train_mae = mean_absolute_error(y_train, train_predictions)
line_train_rmse = root_mean_squared_error(y_train, train_predictions)
line_test_mae = mean_absolute_error(y_test, test_predictions)
line_test_rmse = root_mean_squared_error(y_test, test_predictions)

print('linear train mae =', round(line_train_mae, 4))
print('linear train rmse =', round(line_train_rmse, 4))
print('linear test mae =', round(line_test_mae, 4))
print('linear test rmse =', round(line_test_rmse, 4))


## 一个手写的二次多项式回归

为了保持最小环境可运行，我们用高斯消元手写二次拟合。重点是理解“增加特征表达能力”的效果，而不是记住库接口。

In [ ]:
def solve_linear_system(matrix, values):
    augmented = [row[:] + [value] for row, value in zip(matrix, values)]
    size = len(augmented)
    for pivot in range(size):
        pivot_row = max(range(pivot, size), key=lambda idx: abs(augmented[idx][pivot]))
        augmented[pivot], augmented[pivot_row] = augmented[pivot_row], augmented[pivot]
        pivot_value = augmented[pivot][pivot]
        for column in range(pivot, size + 1):
            augmented[pivot][column] /= pivot_value
        for row_index in range(size):
            if row_index == pivot:
                continue
            factor = augmented[row_index][pivot]
            for column in range(pivot, size + 1):
                augmented[row_index][column] -= factor * augmented[pivot][column]
    return [row[-1] for row in augmented]

def fit_quadratic(xs, ys):
    design = [[1.0, x, x * x] for x in xs]
    matrix = [[0.0] * 3 for _ in range(3)]
    values = [0.0] * 3
    for row, y_value in zip(design, ys):
        for i in range(3):
            values[i] += row[i] * y_value
            for j in range(3):
                matrix[i][j] += row[i] * row[j]
    return solve_linear_system(matrix, values)

def predict_quadratic(x_value, coefficients):
    a0, a1, a2 = coefficients
    return a0 + a1 * x_value + a2 * x_value * x_value

quadratic_coeffs = fit_quadratic(x_train, y_train)
quadratic_train_predictions = [predict_quadratic(x, quadratic_coeffs) for x in x_train]
quadratic_test_predictions = [predict_quadratic(x, quadratic_coeffs) for x in x_test]

quadratic_train_mae = mean_absolute_error(y_train, quadratic_train_predictions)
quadratic_train_rmse = root_mean_squared_error(y_train, quadratic_train_predictions)
quadratic_test_mae = mean_absolute_error(y_test, quadratic_test_predictions)
quadratic_test_rmse = root_mean_squared_error(y_test, quadratic_test_predictions)

print('quadratic coefficients =', [round(value, 3) for value in quadratic_coeffs])
print('quadratic train mae =', round(quadratic_train_mae, 4))
print('quadratic train rmse =', round(quadratic_train_rmse, 4))
print('quadratic test mae =', round(quadratic_test_mae, 4))
print('quadratic test rmse =', round(quadratic_test_rmse, 4))


In [ ]:
comparison_rows = []
for row, line_pred, quad_pred in zip(test_rows, test_predictions, quadratic_test_predictions):
    comparison_rows.append({
        'galaxy_id': row['galaxy_id'],
        'truth': round(row['specz'], 3),
        'linear_pred': round(line_pred, 3),
        'quadratic_pred': round(quad_pred, 3),
        'linear_residual': round(row['specz'] - line_pred, 4),
        'quadratic_residual': round(row['specz'] - quad_pred, 4),
    })

comparison_rows


In [ ]:
outside_example = 0.95
print('Extrapolation warning example for g-r =', outside_example)
print('  linear prediction =', round(predict_line(outside_example, slope, intercept), 3))
print('  quadratic prediction =', round(predict_quadratic(outside_example, quadratic_coeffs), 3))
print('  note: this x value is near or beyond the sparse edge of the tiny teaching sample.')


## 如果安装了 scikit-learn

下面这个单元是可选的。它把最小教学实现与常见生态中的做法连起来。

In [ ]:
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import PolynomialFeatures
except ModuleNotFoundError:
    print('scikit-learn 未安装；已跳过标准库回归示例。')
else:
    x_train_multi = [[row['g_r'], row['r_i']] for row in train_rows]
    x_test_multi = [[row['g_r'], row['r_i']] for row in test_rows]

    ridge_model = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), Ridge(alpha=0.01))
    ridge_model.fit(x_train_multi, y_train)
    ridge_predictions = ridge_model.predict(x_test_multi)
    print('ridge mae =', round(mean_absolute_error(y_test, ridge_predictions), 4))

    forest_model = RandomForestRegressor(n_estimators=50, random_state=42)
    forest_model.fit(x_train_multi, y_train)
    forest_predictions = forest_model.predict(x_test_multi)
    print('random-forest mae =', round(mean_absolute_error(y_test, forest_predictions), 4))


## Algorithm Understanding Bridge

这一段把前面的回归实验压缩成一份可复查的算法理解草稿。它不是新的 record 模板，而是 `Model Experiment Record` 中 `Algorithm Card link`、`Evaluation` 和 `Error Analysis` 的附件。

In [ ]:
RESULTS_DIR = Path('results/part3_algorithm_understanding')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

line_generalization_gap = line_test_mae - line_train_mae
quadratic_generalization_gap = quadratic_test_mae - quadratic_train_mae
worst_linear = sorted(comparison_rows, key=lambda row: abs(row['linear_residual']), reverse=True)
worst_quadratic = sorted(comparison_rows, key=lambda row: abs(row['quadratic_residual']), reverse=True)

regression_bridge = f'''# Ch18 Algorithm Understanding Bridge

## Mean Baseline

- Task role: constant regression baseline.
- Objective intuition: under squared error, the best constant is the training-set mean of the target.
- Diagnostic use: every learned regressor should beat this before it is treated as useful evidence.

## Linear Regression

- Model form: z_hat = a * (g-r) + b.
- Optimized quantity: sum of squared residuals on the training set.
- What is learned: slope a and intercept b.
- Closed-form view: the fitted line is the least-squares projection of the target onto the feature and a constant term.
- Gradient-descent view: repeated small parameter updates would reduce the same squared-error objective.
- Current test MAE: {line_test_mae:.4f}; train-test MAE gap: {line_generalization_gap:.4f}.

## Polynomial Regression

- Model form: z_hat = a0 + a1 * x + a2 * x^2.
- Important point: this is still linear regression after expanding the feature space.
- Benefit: it can represent curvature.
- Risk: near sparse edges of feature space, curvature can become extrapolation rather than evidence.
- Current test MAE: {quadratic_test_mae:.4f}; train-test MAE gap: {quadratic_generalization_gap:.4f}.

## Ridge / Lasso

- Ridge objective: squared error plus an L2 coefficient penalty.
- Lasso objective: squared error plus an L1 coefficient penalty.
- User-fixed quantity: alpha/lambda controls how strongly model complexity is penalized.
- Scientific role: regularization is a bias-variance tradeoff, not a guarantee of physical truth.

## Residual Diagnostics To Attach

- Worst linear residual: {worst_linear[0]['galaxy_id']} with residual {worst_linear[0]['linear_residual']}.
- Worst quadratic residual: {worst_quadratic[0]['galaxy_id']} with residual {worst_quadratic[0]['quadratic_residual']}.
- Required checks: residual vs target, residual vs magnitude or color, residual by galaxy type, and extrapolation near feature-space edges.

## Claim Boundary

This notebook can support a limited claim about how simple color-based regressors behave on a tiny teaching sample. It cannot support a survey-quality photo-z performance claim.
'''

output_path = RESULTS_DIR / 'ch18_algorithm_understanding.md'
output_path.write_text(regression_bridge, encoding='utf-8')
print('wrote', output_path)
print(regression_bridge.splitlines()[0])


## 小结

这个例子展示了一个很关键的节奏：先用简单线性 baseline 建立直觉，再通过误差和残差判断是否值得增加模型表达能力。更复杂的拟合不一定更科学，尤其当它开始靠近样本边界或离开训练范围时。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'line_train_mae': round(line_train_mae, 4),
    'line_test_mae': round(line_test_mae, 4),
    'quadratic_train_mae': round(quadratic_train_mae, 4),
    'quadratic_test_mae': round(quadratic_test_mae, 4),
    'line_test_rmse': round(line_test_rmse, 4),
    'quadratic_test_rmse': round(quadratic_test_rmse, 4),
    'python_version': platform.python_version(),
}

print('Regression snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
